In [19]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def x_y_mnist(num_classes, input_shape):    
    (x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
    
    x_train = x_train.astype("float32") / 255
    x_test = x_test.astype("float32") / 255
    x_train = np.expand_dims(x_train, -1)
    x_test = np.expand_dims(x_test, -1)
    print("x_train shape:", x_train.shape)
    print(x_train.shape[0], "train samples")
    print(x_test.shape[0], "test samples")
    
    y_train = keras.utils.to_categorical(y_train, num_classes)
    y_test = keras.utils.to_categorical(y_test, num_classes)
    return x_train, y_train, x_test, y_test

def x_y_cifar(num_classes, input_shape):
    (x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

    x_train = x_train.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0

    if x_train.shape[1:] != input_shape:
        x_train = np.reshape(x_train, (-1, *input_shape))
        x_test = np.reshape(x_test, (-1, *input_shape))

    print("x_train shape:", x_train.shape)
    print(x_train.shape[0], "train samples")
    print(x_test.shape[0], "test samples")

    y_train = keras.utils.to_categorical(y_train, num_classes)
    y_test = keras.utils.to_categorical(y_test, num_classes)

    return x_train, y_train, x_test, y_test


In [2]:
def train_keras(x_train, y_train, num_classes, input_shape):
    batch_size, epochs = 128, 5
    
    model = keras.Sequential(
        [
            keras.Input(shape=input_shape),
            layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
            layers.MaxPooling2D(pool_size=(2, 2)),
            layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
            layers.MaxPooling2D(pool_size=(2, 2)),
            layers.Flatten(),
            layers.Dropout(0.5),
            layers.Dense(num_classes, activation="softmax"),
        ]
    )
    
    model.summary()
    model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1)

    return model

In [3]:
import tensorflow as tf
import tf2onnx

def keras_to_onnx(model, path):
    model.export(path)
    !python -m tf2onnx.convert --saved-model "{path}" --output "{path}.onnx" --opset 13

In [4]:
import time
import onnxruntime as ort
import tensorflow as tf

def benchmark_model(model, onnx_session, x_data, batch_size, onnx_input_name = None, onnx_output_name = None):
    x_subset = x_data[:1000]  
    num_batches = len(x_subset) // batch_size

    if onnx_input_name:
        onnx_session.run([onnx_output_name], {onnx_input_name: x_subset[:batch_size].astype(np.float32)})
    else:
        model.predict(x_subset[:batch_size], verbose=0)

    start = time.time()
    for i in range(num_batches):
        batch = x_subset[i*batch_size:(i+1)*batch_size]
        if onnx_input_name:
            onnx_session.run([onnx_output_name], {onnx_input_name: batch.astype(np.float32)})
        else:
            model.predict(batch, verbose=0)
    end = time.time()

    avg_time = (end - start) / num_batches
    return avg_time

def models_speed(model, onnx_session, x_test, onnx_input_name, onnx_output_name):
    for bs in [1, 8, 32, 128]:
        keras_time = benchmark_model(model, onnx_session, x_test, bs)
        onnx_time = benchmark_model(model, onnx_session, x_test, bs, onnx_input_name, onnx_output_name)
        print(f"Batch size {bs:3d} | Keras: {keras_time:.6f}s | ONNX: {onnx_time:.6f}s")


In [5]:
def compare_accuracy_softpreds(model, onnx_session, onnx_input_name, onnx_output_name):
    keras_preds = model.predict(x_test, verbose=0)
    keras_acc = np.mean(np.argmax(keras_preds, axis=1) == np.argmax(y_test, axis=1))

    onnx_preds = []
    for i in range(0, len(x_test), 128):
        batch = x_test[i:i+128].astype(np.float32)
        preds = onnx_session.run([onnx_output_name], {onnx_input_name: batch})[0]
        onnx_preds.append(preds)
    onnx_preds = np.vstack(onnx_preds)
    onnx_acc = np.mean(np.argmax(onnx_preds, axis=1) == np.argmax(y_test, axis=1))

    print(f"\nKeras test accuracy: {keras_acc:.4f}")
    print(f"ONNX test accuracy:  {onnx_acc:.4f}")
    
    idxs = np.random.choice(len(x_test), 5, replace=False)
    for i in idxs:
        keras_p = keras_preds[i]
        onnx_p = onnx_preds[i]
        diff = np.abs(keras_p - onnx_p).max()
        print(f"Sample {i}: max abs diff = {diff:.8f}")

In [12]:
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType
import onnx

class DataReader(CalibrationDataReader):
    def __init__(self, data, input_name):
        self.data = data
        self.input_name = input_name
        self.enum_data = None

    def get_next(self):
        if self.enum_data is None:
            self.enum_data = iter([{self.input_name: self.data.astype(np.float32)}])
        return next(self.enum_data, None)

def quantize(onnx_model_path, quantized_model_path, x_test):
    model = onnx.load(onnx_model_path)
    input_name = model.graph.input[0].name

    calibration_data = x_test[:100]
    dr = DataReader(calibration_data, input_name)

    quantize_static(
        model_input=onnx_model_path,
        model_output=quantized_model_path,
        calibration_data_reader=dr,
        quant_format="QDQ",
        weight_type=QuantType.QInt8,
        activation_type=QuantType.QInt8
    )

In [7]:
num_classes, input_shape = 10, (28, 28, 1)

x_train, y_train, x_test, y_test = x_y_mnist(num_classes, input_shape)
mnist_model = train_keras(x_train, y_train, num_classes, input_shape)

x_train shape: (60000, 28, 28, 1)
60000 train samples
10000 test samples


I0000 00:00:1760890241.914966   86799 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-10-19 18:10:41.919495: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2343] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        16,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,826 (136.04 KB)

 Trainable params: 34,826 (136.04 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 12s 26ms/step - accuracy: 0.8910 - loss: 0.3607 - val_accuracy: 0.9782 - val_loss: 0.0826
Epoch 2/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 11s 25ms/step - accuracy: 0.9652 - loss: 0.1138 - val_accuracy: 0.9822 - val_loss: 0.0665
Epoch 3/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 11s 26ms/step - accuracy: 0.9733 - loss: 0.0859 - val_accuracy: 0.9880 - val_loss: 0.0471
Epoch 4/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 11s 27ms/step - accuracy: 0.9779 - loss: 0.0719 - val_accuracy: 0.9893 - val_loss: 0.0407
Epoch 5/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 11s 27ms/step - accuracy: 0.9800 - loss: 0.0630 - val_accuracy: 0.9878 - val_loss: 0.0416


In [8]:
keras_to_onnx(mnist_model, "mnist_model")

INFO:tensorflow:Assets written to: mnist_model/assets


INFO:tensorflow:Assets written to: mnist_model/assets


Saved artifact at 'mnist_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  140270704100944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140270704103248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140270704104592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140270704102480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140270704103632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140270704105168: TensorSpec(shape=(), dtype=tf.resource, name=None)
<frozen runpy>:128: RuntimeWarning: 'tf2onnx.convert' found in sys.modules after import of package 'tf2onnx', but prior to execution of 'tf2onnx.convert'; this may result in unpredictable behaviour
2025-10-19 18:11:44,674 - WARNING - ***IMPORTANT*** Installed protobuf is not cpp accelerated. Conversio

In [10]:
mnist_onnx_session = ort.InferenceSession("mnist_model.onnx", providers=["CPUExecutionProvider"])
mnist_onnx_input_name, mnist_onnx_output_name = mnist_onnx_session.get_inputs()[0].name, mnist_onnx_session.get_outputs()[0].name

models_speed(mnist_model, mnist_onnx_session, x_test, mnist_onnx_input_name, mnist_onnx_output_name)
compare_accuracy_softpreds(mnist_model, mnist_onnx_session, mnist_onnx_input_name, mnist_onnx_output_name)

Batch size   1 | Keras: 0.050468s | ONNX: 0.000059s
Batch size   8 | Keras: 0.049385s | ONNX: 0.000127s
Batch size  32 | Keras: 0.045299s | ONNX: 0.000339s
Batch size 128 | Keras: 0.050117s | ONNX: 0.002134s

Keras test accuracy: 0.9875
ONNX test accuracy:  0.9875
Sample 3524: max abs diff = 0.00000018
Sample 6761: max abs diff = 0.00000012
Sample 9118: max abs diff = 0.00000006
Sample 4423: max abs diff = 0.00000018
Sample 2982: max abs diff = 0.00000006


In [15]:
quantize("mnist_model.onnx", "mnist_model_quantized.onnx", x_test)

mnist_quantized_onnx_session = ort.InferenceSession("mnist_model_quantized.onnx", providers=["CPUExecutionProvider"])
mnist_quantized_onnx_input_name, mnist_quantized_onnx_output_name = mnist_quantized_onnx_session.get_inputs()[0].name, mnist_quantized_onnx_session.get_outputs()[0].name

models_speed(mnist_model, mnist_quantized_onnx_session, x_test, mnist_quantized_onnx_input_name, mnist_quantized_onnx_output_name)
compare_accuracy_softpreds(mnist_model, mnist_quantized_onnx_session, mnist_quantized_onnx_input_name, mnist_quantized_onnx_output_name)

Batch size   1 | Keras: 0.048249s | ONNX: 0.000094s
Batch size   8 | Keras: 0.051324s | ONNX: 0.000333s
Batch size  32 | Keras: 0.042817s | ONNX: 0.001147s
Batch size 128 | Keras: 0.048720s | ONNX: 0.004577s

Keras test accuracy: 0.9875
ONNX test accuracy:  0.9874
Sample 5334: max abs diff = 0.00376344
Sample 9469: max abs diff = 0.00205225
Sample 4813: max abs diff = 0.00291991
Sample 1048: max abs diff = 0.00389916
Sample 2096: max abs diff = 0.00389367


In [17]:
import os

orig_size = os.path.getsize("mnist_model.onnx") / (1024 * 1024)
quant_size = os.path.getsize("mnist_model_quantized.onnx") / (1024 * 1024)

print(f"Non-quantized model size : {orig_size:.2f} MB")
print(f"Quantized model size : {quant_size:.2f} MB")

Non-quantized model size : 0.14 MB
Quantized model size : 0.05 MB


In [23]:
num_classes, input_shape = 10, (32, 32, 3)

x_train, y_train, x_test, y_test = x_y_cifar(num_classes, input_shape)
cifar_model = train_keras(x_train, y_train, num_classes, input_shape)

keras_to_onnx(cifar_model, "cifar_model")

cifar_onnx_session = ort.InferenceSession("cifar_model.onnx", providers=["CPUExecutionProvider"])
cifar_onnx_input_name, cifar_onnx_output_name = cifar_onnx_session.get_inputs()[0].name, cifar_onnx_session.get_outputs()[0].name

models_speed(cifar_model, cifar_onnx_session, x_test, cifar_onnx_input_name, cifar_onnx_output_name)
compare_accuracy_softpreds(cifar_model, cifar_onnx_session, cifar_onnx_input_name, cifar_onnx_output_name)

quantize("cifar_model.onnx", "cifar_model_quantized.onnx", x_test)

cifar_quantized_onnx_session = ort.InferenceSession("cifar_model_quantized.onnx", providers=["CPUExecutionProvider"])
cifar_quantized_onnx_input_name, cifar_quantized_onnx_output_name = cifar_quantized_onnx_session.get_inputs()[0].name, cifar_quantized_onnx_session.get_outputs()[0].name

models_speed(cifar_model, cifar_quantized_onnx_session, x_test, cifar_quantized_onnx_input_name, cifar_quantized_onnx_output_name)
compare_accuracy_softpreds(cifar_model, cifar_quantized_onnx_session, cifar_quantized_onnx_input_name, cifar_quantized_onnx_output_name)

orig_size = os.path.getsize("cifar_model.onnx") / (1024 * 1024)
quant_size = os.path.getsize("cifar_model_quantized.onnx") / (1024 * 1024)

print(f"Non-quantized model size : {orig_size:.2f} MB")
print(f"Quantized model size : {quant_size:.2f} MB")

x_train shape: (50000, 32, 32, 3)
50000 train samples
10000 test samples


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 30, 30, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 15, 15, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 13, 13, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 2304)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 2304)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │        23,050 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 42,442 (165.79 KB)

 Trainable params: 42,442 (165.79 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 15s 40ms/step - accuracy: 0.3714 - loss: 1.7366 - val_accuracy: 0.4926 - val_loss: 1.4389
Epoch 2/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 13s 36ms/step - accuracy: 0.4999 - loss: 1.4131 - val_accuracy: 0.5500 - val_loss: 1.2895
Epoch 3/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 14s 39ms/step - accuracy: 0.5361 - loss: 1.3105 - val_accuracy: 0.5836 - val_loss: 1.2019
Epoch 4/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 13s 36ms/step - accuracy: 0.5643 - loss: 1.2429 - val_accuracy: 0.6062 - val_loss: 1.1563
Epoch 5/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 14s 38ms/step - accuracy: 0.5854 - loss: 1.1939 - val_accuracy: 0.6160 - val_loss: 1.1110
INFO:tensorflow:Assets written to: cifar_model/assets


INFO:tensorflow:Assets written to: cifar_model/assets


Saved artifact at 'cifar_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 3), dtype=tf.float32, name='keras_tensor_24')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  140270690633104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140270690636176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140270690637904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140270690637520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140270690638288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140270690638096: TensorSpec(shape=(), dtype=tf.resource, name=None)
<frozen runpy>:128: RuntimeWarning: 'tf2onnx.convert' found in sys.modules after import of package 'tf2onnx', but prior to execution of 'tf2onnx.convert'; this may result in unpredictable behaviour
2025-10-19 18:28:30,801 - WARNING - ***IMPORTANT*** Installed protobuf is not cpp accelerated. Conver


Keras test accuracy: 0.6086
ONNX test accuracy:  0.6086
Sample 2807: max abs diff = 0.00000012
Sample 6833: max abs diff = 0.00000008
Sample 5360: max abs diff = 0.00000006
Sample 320: max abs diff = 0.00000006
Sample 6393: max abs diff = 0.00000007
Batch size   1 | Keras: 0.048347s | ONNX: 0.000103s
Batch size   8 | Keras: 0.052269s | ONNX: 0.000379s
Batch size  32 | Keras: 0.049706s | ONNX: 0.001304s
Batch size 128 | Keras: 0.059648s | ONNX: 0.006363s

Keras test accuracy: 0.6086
ONNX test accuracy:  0.6070
Sample 1968: max abs diff = 0.00622237
Sample 1188: max abs diff = 0.00713304
Sample 5066: max abs diff = 0.00597185
Sample 4806: max abs diff = 0.00841898
Sample 4235: max abs diff = 0.00523528
Non-quantized model size : 0.17 MB
Quantized model size : 0.06 MB
